# WorkforceIQ - Workforce Intelligence Report

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [2]:
# Load HR dataset
df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
# Dataset overview
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


Rows: 1470
Columns: 35


In [4]:
# Executive workforce snapshot
total_employees = len(df)
attrition_rate = ((df["Attrition"] == "Yes").mean()) * 100
avg_salary = df["MonthlyIncome"].mean()
avg_tenure = df["YearsAtCompany"].mean()

print(f"Total Employees : {total_employees}")
print(f"Attrition Rate  : {attrition_rate:.2f}%")
print(f"Average Salary  : ${avg_salary:,.0f}")
print(f"Average Tenure  : {avg_tenure:.1f} years")


Total Employees : 1470
Attrition Rate  : 16.12%
Average Salary  : $6,503
Average Tenure  : 7.0 years


In [5]:
# Workforce distribution by department
fig = px.histogram(df, x="Department", title="Workforce Distribution by Department")
fig.show()


In [6]:
# Gender distribution
fig = px.pie(df, names="Gender", title="Gender Distribution")
fig.show()


In [7]:
# Employee age distribution
fig = px.histogram(df, x="Age", nbins=20, title="Employee Age Distribution")
fig.show()


In [8]:
# Employee attrition overview
attrition_summary = df.groupby("Attrition").size().reset_index(name="Count")
fig = px.pie(attrition_summary, names="Attrition", values="Count", title="Employee Attrition Overview")
fig.show()


In [9]:
# Attrition by department
attrition_dept = pd.crosstab(df["Department"], df["Attrition"], normalize="index") * 100
attrition_dept = attrition_dept.reset_index()

fig = px.bar(attrition_dept, x="Department", y="Yes", title="Attrition Rate by Department")
fig.show()


In [10]:
# Workforce Health Score
df["WorkforceHealthScore"] = (
    df["JobSatisfaction"]
    + df["WorkLifeBalance"]
    + df["EnvironmentSatisfaction"]
    + df["JobInvolvement"]
) / 16 * 100

df["WorkforceHealthScore"].describe()


count    1470.000000
mean       68.384354
std        11.441492
min        31.250000
25%        62.500000
50%        68.750000
75%        75.000000
max       100.000000
Name: WorkforceHealthScore, dtype: float64

In [11]:
# Workforce Health Score distribution
fig = px.histogram(df, x="WorkforceHealthScore", nbins=20, title="Distribution of Workforce Health Score")
fig.show()


In [12]:
# Workforce Health Score by attrition status
health_attrition = (
    df.groupby("Attrition")["WorkforceHealthScore"]
      .mean()
      .reset_index()
)

fig = px.bar(health_attrition, x="Attrition", y="WorkforceHealthScore",
             title="Workforce Health Score by Attrition Status")
fig.show()


In [13]:
# Employee health segmentation
df["HealthSegment"] = pd.cut(
    df["WorkforceHealthScore"],
    bins=[0, 50, 70, 100],
    labels=["At Risk", "Moderate", "Healthy"]
)

df["HealthSegment"].value_counts()


HealthSegment
Moderate    765
Healthy     570
At Risk     135
Name: count, dtype: int64

In [14]:
# Workforce health segmentation
segment_dist = df["HealthSegment"].value_counts().reset_index()
segment_dist.columns = ["HealthSegment", "Employees"]

fig = px.pie(segment_dist, names="HealthSegment", values="Employees",
             title="Workforce Health Segmentation")
fig.show()


In [15]:
# Attrition across workforce health segments
health_segment_attrition = pd.crosstab(
    df["HealthSegment"],
    df["Attrition"],
    normalize="index"
) * 100

health_segment_attrition


Attrition,No,Yes
HealthSegment,,
At Risk,60.000000,40.000000
Moderate,83.660131,16.339869
Healthy,89.824561,10.175439


In [16]:
# Attrition by workforce health segment
fig = px.bar(
    health_segment_attrition.reset_index(),
    x="HealthSegment",
    y="Yes",
    color="HealthSegment",
    title="Attrition Rate by Workforce Health Segment",
    text_auto=".1f"
)

fig.show()


In [17]:
# Attrition flag
df["AttritionFlag"] = df["Attrition"].map({"No": 0, "Yes": 1})


In [18]:
# Top workforce risk indicators
risk_factors = (
    df.select_dtypes(include=["int64", "float64"])
      .corr()["AttritionFlag"]
      .sort_values()
)

risk_factors


WorkforceHealthScore       -0.199275
TotalWorkingYears          -0.171063
JobLevel                   -0.169105
YearsInCurrentRole         -0.160545
MonthlyIncome              -0.159840
Age                        -0.159205
YearsWithCurrManager       -0.156199
StockOptionLevel           -0.137145
YearsAtCompany             -0.134392
JobInvolvement             -0.130016
JobSatisfaction            -0.103481
EnvironmentSatisfaction    -0.103369
WorkLifeBalance            -0.063939
TrainingTimesLastYear      -0.059478
DailyRate                  -0.056652
RelationshipSatisfaction   -0.045872
YearsSinceLastPromotion    -0.033019
Education                  -0.031373
PercentSalaryHike          -0.013478
EmployeeNumber             -0.010577
HourlyRate                 -0.006846
PerformanceRating           0.002889
MonthlyRate                 0.015170
NumCompaniesWorked          0.043494
DistanceFromHome            0.077924
AttritionFlag               1.000000
EmployeeCount                    NaN
S

In [19]:
# Top retention drivers
top_factors = (
    risk_factors
    .drop(["AttritionFlag"])
    .dropna()
    .head(10)
    .reset_index()
)

top_factors.columns = ["Factor", "Correlation"]

fig = px.bar(
    top_factors,
    x="Correlation",
    y="Factor",
    orientation="h",
    title="Top Retention Drivers"
)

fig.show()


In [20]:
# Department workforce analysis
dept_health = (
    df.groupby("Department")
      .agg({
          "WorkforceHealthScore":"mean",
          "AttritionFlag":"mean"
      })
      .reset_index()
)

dept_health["AttritionFlag"] *= 100
dept_health


,Department,WorkforceHealthScore,AttritionFlag
0,Human Resources,68.452381,19.047619
1,Research & Development,68.359781,13.839750
2,Sales,68.427691,20.627803


In [21]:
# Department risk matrix
fig = px.scatter(
    dept_health,
    x="WorkforceHealthScore",
    y="AttritionFlag",
    size="AttritionFlag",
    color="Department",
    text="Department",
    title="Department Workforce Risk Matrix"
)

fig.show()


In [22]:
# Highest attrition departments
dept_health.sort_values("AttritionFlag", ascending=False)


,Department,WorkforceHealthScore,AttritionFlag
2,Sales,68.427691,20.627803
0,Human Resources,68.452381,19.047619
1,Research & Development,68.359781,13.839750


In [23]:
df.dtypes

Age                            int64
Attrition                        str
BusinessTravel                   str
DailyRate                      int64
Department                       str
DistanceFromHome               int64
Education                      int64
EducationField                   str
EmployeeCount                  int64
EmployeeNumber                 int64
EnvironmentSatisfaction        int64
Gender                           str
HourlyRate                     int64
JobInvolvement                 int64
JobLevel                       int64
JobRole                          str
JobSatisfaction                int64
MaritalStatus                    str
MonthlyIncome                  int64
MonthlyRate                    int64
NumCompaniesWorked             int64
Over18                           str
OverTime                         str
PercentSalaryHike              int64
PerformanceRating              int64
RelationshipSatisfaction       int64
StandardHours                  int64
S